1. Importaciones y configuración inicial

In [1]:
import pandas as pd
import os

os.makedirs("data/processed", exist_ok=True)
ruta_carpeta_csv = r"C:\Users\misab\Documents\IT Academy\Especialitzacio\sprint 13\notebooks\data\raw"

2. Importar csv

In [2]:
#Importar csv lista países

countries = pd.read_csv(os.path.join(ruta_carpeta_csv, "countries.csv"))

countries

,country_code,country_name
0,BE,Belgium
1,DE,Germany
2,EE,Estonia
3,IE,Ireland
4,EL,Greece
5,ES,Spain
6,FR,France
7,HR,Croatia
8,IT,Italy
9,CY,Cyprus


In [3]:
#Obtener lista archivos a procesar

archivos_csv = [archivo for archivo in os.listdir(ruta_carpeta_csv) if archivo.endswith(".csv")]
variables = [os.path.splitext(archivo)[0] for archivo in archivos_csv]
#Eliminar countries:
variables.remove("countries")
variables

['education',
 'gdp_capita',
 'gini',
 'health',
 'housing',
 'inflation',
 'unemployment']

In [4]:
#Importar y procesar csv
def procesar_csv(inicio, fin, lista_variables):
    """ 
    Función para cargar, revisar y filtrar por años los archivos csv.
    Input: año inicio serie, año fin serie, lista de nombres de variables a procesar.
    Output: Mensaje de df completo o no y diccionario con todos los df.
    """
    dicc_df = {}

    for variable in lista_variables:
        print(f"\n>>DATAFRAME: {variable}")
        df = pd.read_csv(os.path.join(ruta_carpeta_csv, f"{variable}.csv"))
        
        print(">>Información inicial:")
        print(df.info())

        print(">>Años disponibles:")
        print(df["year"].unique())

        #Filtrar por inicio-fin
        df_filtrado = df[(df["year"]>=inicio)&(df["year"]<=fin)]

        print(">>Información sobre métrica después de filtrado:")
        print(df_filtrado.iloc[:, 2].describe())

        print(">>AVISOS:")
        if df_filtrado.empty:
            print("ERROR: Este DataFrame está vacío.")
        else:
            #Aviso si faltan años dentro del rango:
            años_esperados = set(range(inicio, fin + 1))
            años_presentes = set(df_filtrado["year"].unique())
            años_faltantes = años_esperados - años_presentes
            if años_faltantes:
                print(f"REVISAR: En este DataFrame faltan los años: {sorted(años_faltantes)}")
            else:
                print(f"CORRECTO: Datos para todos los años entre {inicio} y {fin}.")
        
        df_filtrado = df_filtrado.sort_values(["country", "year"])
        dicc_df[variable] = df_filtrado

    return dicc_df

In [5]:
dicc_df = procesar_csv(2014, 2024, variables)


>>DATAFRAME: education
>>Información inicial:
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   country        500 non-null    str    
 1   year           500 non-null    int64  
 2   early_leavers  491 non-null    float64
dtypes: float64(1), int64(1), str(1)
memory usage: 11.8 KB
None
>>Años disponibles:
[2000 2001 2002 2003 2004 2005 2006 2007 2008 2009 2010 2011 2012 2013
 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024]
>>Información sobre métrica después de filtrado:
count    220.000000
mean       8.496364
std        3.672119
min        2.000000
25%        6.200000
50%        8.100000
75%       10.125000
max       21.900000
Name: early_leavers, dtype: float64
>>AVISOS:
CORRECTO: Datos para todos los años entre 2014 y 2024.

>>DATAFRAME: gdp_capita
>>Información inicial:
<class 'pandas.DataFrame'>
RangeIndex: 520 entries, 0 to 519
Data colu

In [6]:
dicc_df.keys()


dict_keys(['education', 'gdp_capita', 'gini', 'health', 'housing', 'inflation', 'unemployment'])

In [7]:
#Guardar manualmente cada df del diccionario en una variable
education = dicc_df["education"]
gdp_capita = dicc_df["gdp_capita"]
gini = dicc_df["gini"]
health_23 = dicc_df["health"] #tiene aviso de revisar años
housing = dicc_df["housing"]
inflation = dicc_df["inflation"]
unemployment = dicc_df["unemployment"]

3. Interpolación para DataFrames incompletos

In [8]:
#El único df que no tiene todos los años es health

#Crear df para datos 2024:
paises = health_23["country"].unique()
health_2024 = pd.DataFrame({"country": paises, "year": 2024})
#Crear filas para datos de 2024 unidos a df original:
health = pd.concat([health_23, health_2024], ignore_index=True)
health = health.sort_values(["country", "year"])
#Interpolar por país:
health["healthy_life_years"] = (health.groupby("country")["healthy_life_years"]
        .transform(lambda x: x.interpolate(method="linear", limit_direction="both")))

health.head()

,country,year,healthy_life_years
0,AT,2014,57.7
1,AT,2015,58.0
2,AT,2016,57.0
3,AT,2017,57.1
4,AT,2018,56.9


4. Crear un dataframe unificado y guardar csv

In [9]:
#Unificar nombre columna country para simplificar al hacer merge:
dfs = [education, gdp_capita, gini, health, housing, inflation, unemployment]
for df in dfs:
    df.rename(columns={"country": "country_code"}, inplace=True)

In [10]:
#Unir todos los DataFrames en un único Dataframe (formato wide):
df = countries.merge(gini, on="country_code", how="left")
df = df.merge(housing, on=["country_code", "year"], how="left")
df = df.merge(unemployment, on=["country_code", "year"], how="left")
df = df.merge(education, on=["country_code", "year"], how="left")
df = df.merge(health, on=["country_code", "year"], how="left")
df = df.merge(gdp_capita, on=["country_code", "year"], how="left")
df = df.merge(inflation, on=["country_code", "year"], how="left")
df = df.sort_values(["country_code", "year"]).reset_index(drop=True)
df_correlaciones = df

df_correlaciones

,country_code,country_name,year,gini,home_ownership_rate,unemployment_rate,early_leavers,healthy_life_years,gdp_capita_real,inflation_rate
0,AT,Austria,2014,27.6,57.2,6.0,7.0,57.7,43070.0,1.5
1,AT,Austria,2015,27.2,55.7,6.1,7.3,58.0,43200.0,0.8
2,AT,Austria,2016,27.2,55.0,6.5,6.9,57.0,43550.0,1.0
3,AT,Austria,2017,27.9,55.0,5.9,7.4,57.1,44260.0,2.2
4,AT,Austria,2018,26.8,55.4,5.2,7.3,56.9,45140.0,2.1
...,...,...,...,...,...,...,...,...,...,...
215,SK,Slovakia,2020,20.9,92.3,6.7,7.6,56.7,17270.0,2.0
216,SK,Slovakia,2021,21.8,92.9,6.8,7.8,56.8,18320.0,2.8
217,SK,Slovakia,2022,21.2,93.0,6.1,7.4,57.3,18360.0,12.1
218,SK,Slovakia,2023,21.6,93.6,5.8,6.4,57.5,18750.0,11.0


In [11]:
#Guardar csv en carpeta de datos procesados:
output_path = os.path.join("data", "processed", "df_correlaciones.csv")
df_correlaciones.to_csv(output_path, index=False)
print("Guardado con éxito.")

Guardado con éxito.


In [12]:
#Generar un DataFrame formato long para gráficos:
dim_fijas = ["country_code", "country_name", "year"]
variables = ["gini", "home_ownership_rate", "unemployment_rate", "early_leavers", "healthy_life_years", "gdp_capita_real", "inflation_rate"]

df_graficos = df_correlaciones.melt(id_vars=dim_fijas, value_vars=variables, var_name="variable", value_name="valor")
df_graficos

,country_code,country_name,year,variable,valor
0,AT,Austria,2014,gini,27.6
1,AT,Austria,2015,gini,27.2
2,AT,Austria,2016,gini,27.2
3,AT,Austria,2017,gini,27.9
4,AT,Austria,2018,gini,26.8
...,...,...,...,...,...
1535,SK,Slovakia,2020,inflation_rate,2.0
1536,SK,Slovakia,2021,inflation_rate,2.8
1537,SK,Slovakia,2022,inflation_rate,12.1
1538,SK,Slovakia,2023,inflation_rate,11.0


In [13]:
#Guardar csv en carpeta de datos procesados:
output_path = os.path.join("data", "processed", "df_graficos.csv")
df_graficos.to_csv(output_path, index=False)
print("Guardado con éxito.")

Guardado con éxito.
